In [0]:
%run ./01-ATSConfigs

In [0]:
%run ./03-Endpoints

In [0]:
%run ./04-VectorSearch

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from typing import Iterator

In [0]:
@pandas_udf("string")
def parse_pdf(batch_itr : Iterator[pd.Series]) -> Iterator[pd.Series]:

    import pymupdf
    import pymupdf4llm

    def pdf_md(content):
        pdf_txt  = pymupdf.Document(stream = content, filetype = "pdf")
        md_text = pymupdf4llm.to_markdown(pdf_txt)
        return md_text
    
    for x in batch_itr:
        yield x.apply(pdf_md)
    

In [0]:
class SilverProfile:
    def __init__(self):
        spark.conf.set(
            "spark.databricks.delta.changeDataFeed.timestampOutOfRange.enabled", "true"
        )

    def get_start_time(self):
        start_time = (spark.sql(
            f""" select execution_time as start_time
                 from {conf.jobs_metadata_table_name}
                 where job_name = {conf.profile_silver_job_name}
                 order by execution_time desc
            """
        ).first()
        .asDict()['start_time']
        .strftime('%Y-%d-%m %H:%M:%S')
        )
        return start_time

    def get_end_time(self):
        end_time = (spark.sql(f"""select current_timestamp() as end_time""")
                    .first()
                    .asDict()['end_time']
                    .strftime('%Y-%d-%m %H:%M:%S'))
        return end_time
    def get_last_load_date(self):
        load_date = (
            spark.sql(
                f"""select date_add('last_load_date',1) as load_date
                from {conf.jobs_metadata_table_name}
                where job_name = {conf.profile_silver_job_name}
                order by last_load_date desc"""

            ).first()
            .asDict()['load_date']
            .strftime('%Y-%m-%d %H:%M:%S')

        )
        return load_date


    def update_metadata(self,end_time,load_date):
        spark.sql(
            f"""
            insert into {conf.jobs_metadata_table_name}
            values ('{conf.profile_silver_job_name}','{load_date}',
            '{end_time}',
            'Job Execution')"""
        )
    
    def get_prompt(self):
        from datetime import datetime

        current_year = datetime.now().year
        prompt = f"""
        Extract the following fields from the provided resume text and return them as a single JSON object:

        name

        email

        phone

        linkedin_url

        skills (list: include both explicitly listed skills and those that can be reasonably inferred from project descriptions and work experience)

        education (list of objects: degree, institution, year, grade or score if available)

        work_experience (list of objects: company, role, start_date, end_date, years_of_experience, description)
        achievements (list)

        total_experience (sum of the total number of years of experience)

        When extracting skills:
        - Include all professional skills, tools, technologies, programming languages, libraries, frameworks, methodologies, and techniques that are either explicitly listed or can be reasonably inferred from the candidates project descriptions and work experience.
        - For each project or experience, infer relevant skills based on the tools, methods, or practices described, even if they are not explicitly listed in the skills section.
        - Do not include domain-specific knowledge (such as Law, Medicine, Finance, Education, etc.) as a skill unless the candidates degree, job title, or professional background is specifically in that domain.
        - Only include skills that are relevant to the candidates actual education, profession, or demonstrated expertise.
        - Avoid listing general domain topics (like Legal, Healthcare, Business, Education) as skills unless the candidate is specialized or formally trained in that field.
        - Calculate years_of_experience using the work experience start_date and end_date. Assume current year as {current_year} for calculating years_of_experience.
        - Calculate total_experience as sum of all years_of_experience from the work_experience list.

        For each education entry, extract the degree, institution, year, and any grade, score, or percentage if available, regardless of education level.

        Do not include grades, percentages, or scores in the achievements list if they are already present in the education section. Achievements should only include awards, honors, competitions, or recognitions, not academic grades or marks.

        If a field is missing, set its value to null (for strings/URLs) or an empty array (for lists).

        Resume text:
        """
        return prompt

    #read the only changed data from bronze table
    def extract_profiles(self):
        from spark.sql.functions import expr
        start_time = self.get_start_time()
        end_time = self.get_end_time()

        profile_df = (spark.read.option('readChangeFeed','true')
                      .option('startingTimestamp',start_time)
                      .option('endingTimestamp', end_time)
                      .table(f"{conf.profile_bronze_table_name}")
                      )
        parsed_profile_df = (profile_df.withColumn('text_content', parse_pdf('content'))
                             .selectExpr('path as source', "text_content")
                             .write.mode('overwrite')
                             .saveAsTable(f'{conf.profile_silver_table_name}_stg')
                             )
        
        prompt = self.get_prompt()

        profile_extract_df = (
            spark.read.table(f'{conf.profile_silver_table_name}_stg')
            .withColumn('json_content',expr(f"""
                                            ai_query(endpoint => {conf.llm_endpoint_for_chat},
                                            request => ({prompt},text_content))
                                            """))
        )

        profile_extract_df.write.mode('append').saveAsTable(conf.profile_silver_table_name)
        self.update_metadata(end_time,load_date)

    def assert_count(self, table_name, expected_count):
        print(f"Validating record counts in {table_name}...", end='')
        actual_count = spark.read.table(f"{conf.catalog}.{conf.db}.{table_name}").count()
        assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}" 
        print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

    def validate(self, iter):
        import time
        start = int(time.time())
        print(f"\nValidating profile load into silver layer...")
        self.assert_count(conf.profile_silver_table_name, 5 if iter == 1 else 10)
        print(f"Validating profile load into silver layer completed in {int(time.time()) - start} seconds")
        

        

    #Convert binary data to markdown text
    #write the prompt for getting the data in json format
    #call the LLM and pass the the prompt to get the json
    #write the json data to the silver table